<a href="https://colab.research.google.com/github/Marcin19721205/ProcessControl/blob/main/SteamFlowCompensated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Oblicza skompensowany Przepływ Pary przez kryze

Kod może zadziała nawet jeśli iawps nie zostanie importowany na równaniu gazu doskonałego

In [15]:
import math

try:
    from iapws import IAPWS97
except ImportError:
    # Install the iapws library if it's not found
    !pip install iapws
    try:
        from iapws import IAPWS97
    except ImportError:
        raise ImportError("Brak biblioteki iapws. Zainstaluj: pip install iapws")

try:
    from iapws import IAPWS97

    def rho_steam(P_bar_abs, T_C):
        st = IAPWS97(P=P_bar_abs / 10.0, T=T_C + 273.15)  # P w MPa, T w K
        return 1.0 / st.v                                # kg/m3

except ImportError:
    def rho_steam(P_bar_abs, T_C):
        R = 461.5                                        # J/(kg*K), para wodna
        return P_bar_abs * 1e5 / (R * (T_C + 273.15))    # kg/m3, przybliżenie gazu doskonałego


def eps_orifice(beta, dp_Pa, P1_abs_Pa, kappa=1.30):
    p2_p1 = (P1_abs_Pa - dp_Pa) / P1_abs_Pa
    return 1.0 - (0.351 + 0.256 * beta**4 + 0.93 * beta**8) * (1.0 - p2_p1**(1.0 / kappa))


# ---------------- dane kryzy / rurociągu ----------------

D_mm = 691.0              # średnica wewnętrzna rurociągu
d_mm = 321.37              # średnica otworu kryzy
Cd = 0.61                 # współczynnik wypływu z danych kryzy / ISO
kappa = 1.306              # wykładnik izentropowy pary, orientacyjnie
P_atm_bar = 1.01325       # ciśnienie atmosferyczne

# ---------------- warunki referencyjne ----------------

P_ref_barg = 2.4
T_ref_C = 178.0

# ---------------- warunki aktualne ----------------

P_act_barg = 2.4
T_act_C = 178.0
dp_mbar = 150.0

# ---------------- przeliczenia jednostek ----------------

D = D_mm / 1000.0
d = d_mm / 1000.0
beta = d / D
A0 = math.pi * d**2 / 4.0

dp_Pa = dp_mbar * 100.0

P_ref_abs_bar = P_ref_barg + P_atm_bar
P_act_abs_bar = P_act_barg + P_atm_bar

P_ref_abs_Pa = P_ref_abs_bar * 1e5
P_act_abs_Pa = P_act_abs_bar * 1e5

# ---------------- gęstości pary ----------------

rho_ref = rho_steam(P_ref_abs_bar, T_ref_C)
rho_act = rho_steam(P_act_abs_bar, T_act_C)

# ---------------- współczynniki ekspansji ----------------

eps_ref = eps_orifice(beta, dp_Pa, P_ref_abs_Pa, kappa)
eps_act = eps_orifice(beta, dp_Pa, P_act_abs_Pa, kappa)

# ---------------- przepływ surowy: liczony dla warunków referencyjnych ----------------

m_raw_kg_s = (
    Cd
    * eps_ref
    * A0
    / math.sqrt(1.0 - beta**4)
    * math.sqrt(2.0 * rho_ref * dp_Pa)
)

# ---------------- kompensacja P/T ----------------

m_comp_kg_s = m_raw_kg_s * (eps_act / eps_ref) * math.sqrt(rho_act / rho_ref)

# ---------------- kontrolnie: bezpośrednie obliczenie z warunków aktualnych ----------------

m_direct_kg_s = (
    Cd
    * eps_act
    * A0
    / math.sqrt(1.0 - beta**4)
    * math.sqrt(2.0 * rho_act * dp_Pa)
)

# ---------------- kg/s -> t/h ----------------

Fraw_t_h = m_raw_kg_s * 3.6
Fcomp_t_h = m_comp_kg_s * 3.6
Fdirect_t_h = m_direct_kg_s * 3.6

print("beta =", beta)
print("rho_ref [kg/m3] =", rho_ref)
print("rho_act [kg/m3] =", rho_act)
print("eps_ref =", eps_ref)
print("eps_act =", eps_act)
print("Fraw [t/h] =", Fraw_t_h)
print("Fcomp [t/h] =", Fcomp_t_h)
print("Fdirect [t/h] =", Fdirect_t_h)
print("korekta P/T =", Fcomp_t_h / Fraw_t_h)

beta = 0.4650795947901592
rho_ref [kg/m3] = 1.6764494547936195
rho_act [kg/m3] = 1.6764494547936195
eps_ref = 0.9876530599503215
eps_act = 0.9876530599503215
Fraw [t/h] = 40.41083118192063
Fcomp [t/h] = 40.41083118192063
Fdirect [t/h] = 40.41083118192063
korekta P/T = 1.0


In [11]:
import math

try:
    from iapws import IAPWS97
except ImportError:
    # Install the iapws library if it's not found
    !pip install iapws
    try:
        from iapws import IAPWS97
    except ImportError:
        raise ImportError("Brak biblioteki iapws. Zainstaluj: pip install iapws")


# ---------------- DANE POMIAROWE ----------------

P_barg = 2.40
T_C = 178.00
dp_mbar = 150.0

D_mm = 691.0
d_mm = 321.37

Cd = 0.61
kappa = 1.306
Patm_bar = 1.01325

Pref_barg = 2.40
Tref_C = 178.00


# ---------------- FUNKCJE ----------------

def steam_props(P_bar_abs, T_C):
    st = IAPWS97(P=P_bar_abs / 10.0, T=T_C + 273.15)
    rho = 1.0 / st.v
    h = st.h
    return rho, h


def eps_orifice(beta, dp_bar, P1_abs_bar, kappa):
    P2_abs_bar = P1_abs_bar - dp_bar

    if P2_abs_bar <= 0:
        raise ValueError("Błąd: dp większe niż ciśnienie absolutne przed kryzą.")

    r = P2_abs_bar / P1_abs_bar

    epsilon = 1.0 - (0.351 + 0.256 * beta**4 + 0.93 * beta**8) * (1.0 - r**(1.0 / kappa))

    return epsilon


# ---------------- PRZELICZENIA ----------------

P_abs_bar = P_barg + Patm_bar
Pref_abs_bar = Pref_barg + Patm_bar

dp_bar = dp_mbar / 1000.0
dp_Pa = dp_mbar * 100.0
pierw_dp = math.sqrt(dp_mbar)

D = D_mm / 1000.0
d = d_mm / 1000.0

beta = d / D
Ao = math.pi * d**2 / 4.0


# ---------------- WŁAŚCIWOŚCI PARY IAPWS ----------------

rho_act, h_act = steam_props(P_abs_bar, T_C)
rho_ref, h_ref = steam_props(Pref_abs_bar, Tref_C)


# ---------------- EPSILON ----------------

epsilon_act = eps_orifice(beta, dp_bar, P_abs_bar, kappa)
epsilon_ref = eps_orifice(beta, dp_bar, Pref_abs_bar, kappa)

K_epsilon = epsilon_act / epsilon_ref


# ---------------- STAŁE PRZEPŁYWOWE ----------------

Kp_geom = 3.6 * Cd * Ao * math.sqrt((2.0 * 100.0) / (1.0 - beta**4))
Kp_ref = Kp_geom * epsilon_ref * math.sqrt(rho_ref)


# ---------------- PRZEPŁYW SUROWY ----------------

m_raw_ref_t_h = Kp_ref * pierw_dp


# ---------------- KOREKTY ----------------

K_rho = math.sqrt(rho_act / rho_ref)
K_h = h_act / h_ref

K_total = K_epsilon * K_rho * K_h

m_corr_regulator_t_h = m_raw_ref_t_h * K_total
m_corr_regulator_kg_s = m_corr_regulator_t_h / 3.6


# ---------------- KONTROLA BEZPOŚREDNIA ----------------

m_act_direct_t_h = Kp_geom * epsilon_act * pierw_dp * math.sqrt(rho_act)
m_eq_check_t_h = m_act_direct_t_h * K_h

blad_check = m_corr_regulator_t_h - m_eq_check_t_h


# ---------------- STRUMIEŃ ENERGII ----------------

Q_heat_act_MW = (m_act_direct_t_h / 3.6) * h_act / 1000.0
Q_heat_ref_eq_MW = (m_corr_regulator_t_h / 3.6) * h_ref / 1000.0


# ---------------- FRAW NIEKOMPENSOWANY ----------------

K_Fraw = 3.6 * Cd * epsilon_ref * (math.pi / 4.0) * (d_mm / 1000.0)**2 * math.sqrt(
    (2.0 * 100.0 * rho_ref) / (1.0 - (d_mm / D_mm)**4)
)

Fraw_t_h = K_Fraw * pierw_dp


# ---------------- WYNIKI ----------------

print("\n--- DANE WEJŚCIOWE ---")
print(f"P act             = {P_barg:.3f} barg")
print(f"P act abs         = {P_abs_bar:.3f} bar abs")
print(f"T act             = {T_C:.3f} degC")
print(f"Pref              = {Pref_barg:.3f} barg")
print(f"Pref abs          = {Pref_abs_bar:.3f} bar abs")
print(f"Tref              = {Tref_C:.3f} degC")
print(f"dp                = {dp_mbar:.3f} mbar")
print(f"pierw_dp          = {pierw_dp:.6f} sqrt(mbar)")
print(f"D                 = {D_mm:.3f} mm")
print(f"d                 = {d_mm:.3f} mm")
print(f"beta              = {beta:.12f}")

print("\n--- WŁAŚCIWOŚCI PARY IAPWS ---")
print(f"rho act           = {rho_act:.12f} kg/m3")
print(f"rho ref           = {rho_ref:.12f} kg/m3")
print(f"h act             = {h_act:.12f} kJ/kg")
print(f"h ref             = {h_ref:.12f} kJ/kg")

print("\n--- EPSILON ---")
print(f"epsilon act       = {epsilon_act:.12f}")
print(f"epsilon ref       = {epsilon_ref:.12f}")
print(f"K epsilon         = {K_epsilon:.12f}")

print("\n--- KOREKTY REGULATOROWE ---")
print(f"K rho             = {K_rho:.12f}")
print(f"K h               = {K_h:.12f}")
print(f"K total           = {K_total:.12f}")

print("\n--- STAŁE PRZEPŁYWOWE ---")
print(f"Kp geom           = {Kp_geom:.12f}")
print(f"Kp ref            = {Kp_ref:.12f} t/h/sqrt(mbar)")

print("\n--- PRZEPŁYW DLA REGULATORA ---")
print(f"m raw ref         = {m_raw_ref_t_h:.12f} t/h")
print(f"m corr regulator  = {m_corr_regulator_t_h:.12f} t/h")
print(f"m corr regulator  = {m_corr_regulator_kg_s:.12f} kg/s")

print("\n--- KONTROLA OBLICZENIA ---")
print(f"m act direct      = {m_act_direct_t_h:.12f} t/h")
print(f"m eq check        = {m_eq_check_t_h:.12f} t/h")
print(f"blad check        = {blad_check:.12e} t/h")

print("\n--- STRUMIEŃ ENERGII ---")
print(f"Q heat act        = {Q_heat_act_MW:.12f} MW")
print(f"Q heat ref eq     = {Q_heat_ref_eq_MW:.12f} MW")

print("\n--- FRAW NIEKOMPENSOWANY ---")
print("Wzor: Fraw = K_Fraw * pierw_dp")
print(f"K_Fraw            = {K_Fraw:.12f} t/h/sqrt(mbar)")
print(f"pierw_dp          = {pierw_dp:.12f} sqrt(mbar)")
print(f"Fraw              = {Fraw_t_h:.12f} t/h")


--- DANE WEJŚCIOWE ---
P act             = 2.400 barg
P act abs         = 3.413 bar abs
T act             = 178.000 degC
Pref              = 2.400 barg
Pref abs          = 3.413 bar abs
Tref              = 178.000 degC
dp                = 150.000 mbar
pierw_dp          = 12.247449 sqrt(mbar)
D                 = 691.000 mm
d                 = 321.370 mm
beta              = 0.465079594790

--- WŁAŚCIWOŚCI PARY IAPWS ---
rho act           = 1.676449454794 kg/m3
rho ref           = 1.676449454794 kg/m3
h act             = 2817.959032033837 kJ/kg
h ref             = 2817.959032033837 kJ/kg

--- EPSILON ---
epsilon act       = 0.987653059950
epsilon ref       = 0.987653059950
K epsilon         = 1.000000000000

--- KOREKTY REGULATOROWE ---
K rho             = 1.000000000000
K h               = 1.000000000000
K total           = 1.000000000000

--- STAŁE PRZEPŁYWOWE ---
Kp geom           = 2.580194874249
Kp ref            = 3.299530549249 t/h/sqrt(mbar)

--- PRZEPŁYW DLA REGULATORA ---
m raw